In [0]:

Step 1 — Total cores
4 executors × 4 cores = 16 cores

Step 2 — Partition size
File size          = 10 GB = 10240 MB
defaultParallelism = 16 cores

Partition size = min(128MB, 10240/16)
               = min(128MB, 640MB)
               = 128MB


Step 3 — Number of partitions
Partitions = file size / partition size
           = 10240 / 128
           = 100 partitions

In [0]:
PROJECT = "sports_analytics"
ENV     = "dev"

BASE = f"/Volumes/workspace/default/gangadhar_test/{PROJECT}"



LANDING_PATH = f"{BASE}/landing"   # raw API JSON files
BRONZE_PATH  = f"{BASE}/bronze"    # raw Delta
SILVER_PATH  = f"{BASE}/silver"    # clean Delta
GOLD_PATH    = f"{BASE}/gold"      # metrics Delta
LOG_PATH     = f"{BASE}/logs"      # pipeline logs

# ── Landing zone paths ─────────────────────────
LANDING_EVENTS    = f"{LANDING_PATH}/events"
LANDING_TEAMS     = f"{LANDING_PATH}/teams"
LANDING_STANDINGS = f"{LANDING_PATH}/standings"
LANDING_PLAYERS   = f"{LANDING_PATH}/players"

# ── Bronze paths ───────────────────────────────
BRONZE_EVENTS    = f"{BRONZE_PATH}/events"
BRONZE_TEAMS     = f"{BRONZE_PATH}/teams"
BRONZE_STANDINGS = f"{BRONZE_PATH}/standings"
BRONZE_PLAYERS   = f"{BRONZE_PATH}/players"

# ── Silver paths ───────────────────────────────
SILVER_EVENTS    = f"{SILVER_PATH}/events"
SILVER_STANDINGS = f"{SILVER_PATH}/standings"
SILVER_PLAYERS   = f"{SILVER_PATH}/players"

# ── Gold paths ─────────────────────────────────
GOLD_LEAGUE_STANDINGS  = f"{GOLD_PATH}/league_standings"
GOLD_TEAM_PERFORMANCE  = f"{GOLD_PATH}/team_performance"
GOLD_TOP_SCORERS       = f"{GOLD_PATH}/top_scorers"
GOLD_MATCH_SUMMARY     = f"{GOLD_PATH}/match_summary"
GOLD_DAILY_MIS         = f"{GOLD_PATH}/daily_mis"

# ── API settings ───────────────────────────────
API_BASE_URL = "https://www.thesportsdb.com/api/v1/json/3"


# League IDs (TheSportsDB)
LEAGUES = {
    "EPL":         "4328",  # English Premier League
    "La_Liga":     "4335",  # Spanish La Liga
    "Champions_League": "4480",  # UEFA Champions League
    "IPL":         "4391",  # Indian Premier League
    "NBA":         "4387",  # Basketball
}


all_paths = [
    LANDING_EVENTS, LANDING_TEAMS,
    LANDING_STANDINGS, LANDING_PLAYERS,
    BRONZE_EVENTS, BRONZE_TEAMS,
    BRONZE_STANDINGS, BRONZE_PLAYERS,
    SILVER_EVENTS, SILVER_STANDINGS,
    SILVER_PLAYERS,
    GOLD_LEAGUE_STANDINGS, GOLD_TEAM_PERFORMANCE,
    GOLD_TOP_SCORERS, GOLD_MATCH_SUMMARY,
    GOLD_DAILY_MIS,
    LOG_PATH
]

print("Clearing existing data...")

try:
    dbutils.fs.rm(BASE, recurse=True)
    print(f"Removed: {BASE} ✅")
except:
    print(f"  Nothing to remove")

for path in all_paths:
    dbutils.fs.mkdirs(path)

In [0]:
import requests
import json
import datetime
import time

batch_id   = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
batch_date = datetime.datetime.now().strftime("%Y-%m-%d")

def fetch_api(endpoint, params=""):
    url = f"{API_BASE_URL}/{endpoint}{params}"
    print(f"  Calling: {url}")
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            return response.json()
        else:
            print(f"  Error: {response.status_code}")
            return None
    except Exception as e:
        print(f"  Exception: {str(e)}")
        return None

# ── Clear folder before saving ─────────────────
def clear_folder(path):
    try:
        files = dbutils.fs.ls(path)
        for f in files:
            dbutils.fs.rm(f.path)
            print(f" Deleted: {f.name}")
    except:
        pass  # folder empty or not found

# ── Save to landing zone ───────────────────────
def save_to_landing(data, path, filename):
    # clear_folder(path)
    full_path = f"{path}/{filename}"
    with open(f"{full_path}", "w") as f:
        json.dump(data, f, indent=2)

    print(f"  Saved: {filename} ✅")
    return full_path

print("\n1. Fetching EPL Recent Events...")

epl_events = fetch_api(
    "eventspastleague.php",
    f"?id={LEAGUES['EPL']}"
)

if epl_events:
    epl_events["_batch_id"]   = batch_id
    epl_events["_batch_date"] = batch_date
    epl_events["_league"]     = "EPL"
    epl_events["_fetched_at"] = datetime.datetime.now().isoformat()

    save_to_landing(
        epl_events,
        LANDING_EVENTS,
        f"epl_events_{batch_id}.json"
    )
    print(f"  Events found: "
          f"{len(epl_events.get('events', []))}")
else:
    print("No data returned")

In [0]:
# ── Fetch 2: La Liga Events ────────────────────
print("\n2. Fetching La Liga Recent Events...")

laliga_events = fetch_api(
    f"eventspastleague.php",
    f"?id={LEAGUES['La_Liga']}"
)

if laliga_events:
    laliga_events["_batch_id"]   = batch_id
    laliga_events["_batch_date"] = batch_date
    laliga_events["_league"]     = "La_Liga"
    laliga_events["_fetched_at"] = datetime.datetime \
        .now().isoformat()

    save_to_landing(
        laliga_events,
        LANDING_EVENTS,
        f"laliga_events_{batch_id}.json"
    )
    print(f"  Events found: "
          f"{len(laliga_events.get('events', []))}")

In [0]:
# ── Fetch 3: EPL Teams ─────────────────────────
print("\n3. Fetching EPL Teams...")

epl_teams = fetch_api(
    f"lookup_all_teams.php",
    f"?id={LEAGUES['EPL']}"
)

if epl_teams:
    epl_teams["_batch_id"]   = batch_id
    epl_teams["_batch_date"] = batch_date
    epl_teams["_league"]     = "EPL"
    epl_teams["_fetched_at"] = datetime.datetime \
        .now().isoformat()

    save_to_landing(
        epl_teams,
        LANDING_TEAMS,
        f"epl_teams_{batch_id}.json"
    )
    print(f"  Teams found: "
          f"{len(epl_teams.get('teams', []))}")

time.sleep(1)

In [0]:
# ── Fetch 4: EPL Standings ─────────────────────
print("\n4. Fetching EPL Standings...")

epl_standings = fetch_api(
    f"lookuptable.php",
    f"?l={LEAGUES['EPL']}&s=2023-2024"
)

if epl_standings:
    epl_standings["_batch_id"]   = batch_id
    epl_standings["_batch_date"] = batch_date
    epl_standings["_league"]     = "EPL"
    epl_standings["_fetched_at"] = datetime.datetime \
        .now().isoformat()

    save_to_landing(
        epl_standings,
        LANDING_STANDINGS,
        f"epl_standings_{batch_id}.json"
    )
    table = epl_standings.get("table", [])
    print(f"  Teams in standings: {len(table)}")

time.sleep(1)

In [0]:
# ── Fetch 5: Upcoming EPL Events ──────────────
print("\n5. Fetching Upcoming EPL Events...")

upcoming = fetch_api(
    f"eventsnextleague.php",
    f"?id={LEAGUES['EPL']}"
)

if upcoming:
    upcoming["_batch_id"]   = batch_id
    upcoming["_batch_date"] = batch_date
    upcoming["_league"]     = "EPL"
    upcoming["_type"]       = "upcoming"
    upcoming["_fetched_at"] = datetime.datetime \
        .now().isoformat()

    save_to_landing(
        upcoming,
        LANDING_EVENTS,
        f"epl_upcoming_{batch_id}.json"
    )
    print(f"  Upcoming events: "
          f"{len(upcoming.get('events', []))}")

time.sleep(1)

In [0]:
# ── Fetch 6: Champions League Events ──────────
print("\n6. Fetching Champions League Events...")

ucl_events = fetch_api(
    f"eventspastleague.php",
    f"?id={LEAGUES['Champions_League']}"
)

if ucl_events:
    ucl_events["_batch_id"]   = batch_id
    ucl_events["_batch_date"] = batch_date
    ucl_events["_league"]     = "Champions_League"
    ucl_events["_fetched_at"] = datetime.datetime \
        .now().isoformat()

    save_to_landing(
        ucl_events,
        LANDING_EVENTS,
        f"ucl_events_{batch_id}.json"
    )
    print(f"  UCL events: "
          f"{len(ucl_events.get('events', []))}")

# ── Verify landing zone ────────────────────────
print("\n7. Verifying Landing Zone...")

print("\nEvents files:")
for f in dbutils.fs.ls(LANDING_EVENTS):
    print(f"  {f.name} → {f.size} bytes")

print("\nTeams files:")
for f in dbutils.fs.ls(LANDING_TEAMS):
    print(f"  {f.name} → {f.size} bytes")

print("\nStandings files:")
for f in dbutils.fs.ls(LANDING_STANDINGS):
    print(f"  {f.name} → {f.size} bytes")

print("\n" + "="*55)
print("01_FETCH_API_DATA COMPLETE ✅")
print(f"  Batch ID: {batch_id}")
print(f"  Files saved to landing zone ✅")
print("="*55)

In [0]:
# ============================================
# 02_BRONZE LAYER
# Read JSON from landing zone
# Save as Delta tables
# Add metadata columns
# ============================================

from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
import datetime

batch_id   = datetime.datetime.now() \
    .strftime("%Y%m%d%H%M%S")
batch_date = datetime.datetime.now() \
    .strftime("%Y-%m-%d")

print("="*55)
print("02_BRONZE LAYER")
print("="*55)

def add_metadata(df, source):
    return df \
        .withColumn("_source",
                    lit(source)) \
        .withColumn("_batch_id",
                    lit(batch_id)) \
        .withColumn("_batch_date",
                    lit(batch_date)) \
        .withColumn("_ingested_at",
                    current_timestamp())

def log(task, status, msg, records=0):
    data = [(task, status, msg, records,
             batch_id, batch_date,
             datetime.datetime.now()
             .strftime("%Y-%m-%d %H:%M:%S"))]
    spark.createDataFrame(
        data,
        ["task","status","message",
         "records","batch_id",
         "batch_date","log_time"]
    ).write \
     .format("delta") \
     .mode("append") \
     .save(LOG_PATH)
    print(f"  [{status}] {msg}")

In [0]:
# ── Bronze Events ─────────────────────────────
print("\n1. Ingesting Events...")
log("bronze_events", "STARTED",
    "Reading event JSON files")

# Read all JSON files from landing
events_raw = spark.read \
    .format("json") \
    .option("multiLine", "true") \
    .load(LANDING_EVENTS)

print(f"  Raw columns: {events_raw.columns}")
print(f"  Raw count:   {events_raw.count()}")
events_raw.show(truncate=False)

# Explode events array
# JSON has: {"events": [...], "_batch_id": "..."}
if "events" in events_raw.columns:
    events_exploded = events_raw \
        .select(
            explode(col("events")).alias("event"),
            col("_batch_id"),
            col("_batch_date"),
            col("_league"),
            col("_fetched_at")
        ) \
        .select(
            "event.*",
            "_batch_id",
            "_batch_date",
            "_league",
            "_fetched_at"
        )

    events_bronze = add_metadata(
        events_exploded,
        "THESPORTSDB_API"
    )

    # Merge to avoid duplicates
    if DeltaTable.isDeltaTable(
        spark, BRONZE_EVENTS
    ):
        bronze_tbl = DeltaTable.forPath(
            spark, BRONZE_EVENTS
        )
        bronze_tbl.alias("t") \
            .merge(
                events_bronze.alias("s"),
                "t.idEvent = s.idEvent"
            ) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()
        print("  Merged ✅")
    else:
        events_bronze.write \
            .format("delta") \
            .mode("overwrite") \
            .save(BRONZE_EVENTS)
        print("  Created ✅")

    count = spark.read \
        .format("delta") \
        .load(BRONZE_EVENTS) \
        .count()

    log("bronze_events", "SUCCESS",
        "Events ingested", count)
    print(f"  Bronze events: {count}")

In [0]:
# ── Bronze Standings ──────────────────────────
print("\n2. Ingesting Standings...")

standings_raw = spark.read \
    .format("json") \
    .option("multiLine", "true") \
    .load(LANDING_STANDINGS)

if "table" in standings_raw.columns:
    standings_exploded = standings_raw \
        .select(
            explode(col("table")).alias("standing"),
            col("_batch_id"),
            col("_batch_date"),
            col("_league")
        ) \
        .select(
            "standing.*",
            "_batch_id",
            "_batch_date",
            "_league"
        )

    standings_bronze = add_metadata(
        standings_exploded,
        "THESPORTSDB_API"
    )

    standings_bronze.write \
        .format("delta") \
        .mode("overwrite") \
        .save(BRONZE_STANDINGS)

    count = standings_bronze.count()
    log("bronze_standings", "SUCCESS",
        "Standings ingested", count)
    print(f"  Bronze standings: {count}")
    standings_bronze.show(5)

In [0]:
# ── Bronze Teams ──────────────────────────────
print("\n3. Ingesting Teams...")

teams_raw = spark.read \
    .format("json") \
    .option("multiLine", "true") \
    .load(LANDING_TEAMS)

if "teams" in teams_raw.columns:
    teams_exploded = teams_raw \
        .select(
            explode(col("teams")).alias("team"),
            col("_batch_id"),
            col("_batch_date"),
            col("_league")
        ) \
        .select(
            "team.*",
            "_batch_id",
            "_batch_date",
            "_league"
        )

    teams_bronze = add_metadata(
        teams_exploded,
        "THESPORTSDB_API"
    )

    if DeltaTable.isDeltaTable(
        spark, BRONZE_TEAMS
    ):
        bronze_tbl = DeltaTable.forPath(
            spark, BRONZE_TEAMS
        )
        bronze_tbl.alias("t") \
            .merge(
                teams_bronze.alias("s"),
                "t.idTeam = s.idTeam"
            ) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()
    else:
        teams_bronze.write \
            .format("delta") \
            .mode("overwrite") \
            .save(BRONZE_TEAMS)

    count = spark.read \
        .format("delta") \
        .load(BRONZE_TEAMS) \
        .count()

    log("bronze_teams", "SUCCESS",
        "Teams ingested", count)
    print(f"  Bronze teams: {count}")
    teams_bronze.select(
        "idTeam", "strTeam",
        "strLeague", "intFormedYear"
    ).show(5)

print("\n" + "="*55)
print("02_BRONZE LAYER COMPLETE ✅")
print("="*55)

In [0]:
# ============================================
# 03_SILVER LAYER
# Clean + Transform + Enrich
# Optimization: PARTITIONED by league + date
# ============================================

from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *
import datetime

batch_id   = datetime.datetime.now() \
    .strftime("%Y%m%d%H%M%S")
batch_date = datetime.datetime.now() \
    .strftime("%Y-%m-%d")

print("="*55)
print("03_SILVER LAYER")
print("="*55)

def log(task, status, msg, records=0):
    data = [(task, status, msg, records,
             batch_id, batch_date,
             datetime.datetime.now()
             .strftime("%Y-%m-%d %H:%M:%S"))]
    spark.createDataFrame(
        data,
        ["task","status","message",
         "records","batch_id",
         "batch_date","log_time"]
    ).write \
     .format("delta") \
     .mode("append") \
     .save(LOG_PATH)
    print(f"  [{status}] {msg}")

In [0]:
# ── Read Bronze ───────────────────────────────
print("\n1. Reading Bronze tables...")

events_bronze   = spark.read \
    .format("delta") \
    .load(BRONZE_EVENTS)
standings_bronze = spark.read \
    .format("delta") \
    .load(BRONZE_STANDINGS)
teams_bronze    = spark.read \
    .format("delta") \
    .load(BRONZE_TEAMS)

print(f"  Events:    {events_bronze.count()}")
print(f"  Standings: {standings_bronze.count()}")
print(f"  Teams:     {teams_bronze.count()}")

print("\nEvents schema:")
events_bronze.printSchema()

In [0]:
# ── Silver Events ─────────────────────────────
print("\n2. Cleaning Events...")

# Select and clean key columns
events_silver = events_bronze \
    .select(
        col("idEvent").alias("event_id"),
        col("strEvent").alias("event_name"),
        col("strLeague").alias("league"),
        col("strSeason").alias("season"),
        col("strHomeTeam").alias("home_team"),
        col("strAwayTeam").alias("away_team"),
        col("intHomeScore").alias("home_score"),
        col("intAwayScore").alias("away_score"),
        col("dateEvent").alias("event_date"),
        col("strTime").alias("event_time"),
        col("strVenue").alias("venue"),
        col("strStatus").alias("status"),
        col("strPostponed").alias("postponed"),
        col("_league"),
        col("_batch_date")
    ) \
    .filter(col("event_id").isNotNull()) \
    .filter(col("event_date").isNotNull()) \
    .withColumn("event_date",
        to_date("event_date")) \
    .withColumn("event_year",
        year("event_date")) \
    .withColumn("event_month",
        month("event_date")) \
    .withColumn("home_score",
        col("home_score").cast(IntegerType())) \
    .withColumn("away_score",
        col("away_score").cast(IntegerType())) \
    .withColumn("total_goals",
        col("home_score") + col("away_score")) \
    .withColumn("result",
        when(col("home_score") >
             col("away_score"), "HOME_WIN")
        .when(col("home_score") <
              col("away_score"), "AWAY_WIN")
        .when(col("home_score") ==
              col("away_score"), "DRAW")
        .otherwise("UNKNOWN")
    ) \
    .withColumn("is_high_scoring",
        when(col("total_goals") >= 4, True)
        .otherwise(False)
    ) \
    .withColumn("_processed_at",
                current_timestamp()) \
    .dropDuplicates(["event_id"])

print(f"  Silver events: {events_silver.count()}")

# Write partitioned by league + event_year
events_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("_league", "event_year") \
    .save(SILVER_EVENTS)

print("  Partitioned by league + year ✅")
log("silver_events", "SUCCESS",
    "Events cleaned",
    events_silver.count())
events_silver.show(5)

In [0]:
# ── Silver Standings ──────────────────────────
print("\n3. Cleaning Standings...")

standings_silver = standings_bronze \
    .select(
        col("idStanding").alias("standing_id"),
        col("idTeam").alias("team_id"),
        col("strTeam").alias("team_name"),
        col("strForm").alias("form"),
        col("intPlayed").alias("played"),
        col("intWin").alias("wins"),
        col("intDraw").alias("draws"),
        col("intLoss").alias("losses"),
        col("intGoalsFor").alias("goals_for"),
        col("intGoalsAgainst")
        .alias("goals_against"),
        col("intGoalDifference")
        .alias("goal_difference"),
        col("intPoints").alias("points"),
        col("_league"),
        col("_batch_date")
    ) \
    .filter(col("team_id").isNotNull()) \
    .withColumn("played",
        col("played").cast(IntegerType())) \
    .withColumn("wins",
        col("wins").cast(IntegerType())) \
    .withColumn("draws",
        col("draws").cast(IntegerType())) \
    .withColumn("losses",
        col("losses").cast(IntegerType())) \
    .withColumn("goals_for",
        col("goals_for").cast(IntegerType())) \
    .withColumn("goals_against",
        col("goals_against")
        .cast(IntegerType())) \
    .withColumn("points",
        col("points").cast(IntegerType())) \
    .withColumn("win_rate",
        when(col("played") > 0,
             round(col("wins") /
                   col("played") * 100, 2))
        .otherwise(0)
    ) \
    .withColumn("goals_per_game",
        when(col("played") > 0,
             round(col("goals_for") /
                   col("played"), 2))
        .otherwise(0)
    ) \
    .withColumn("_processed_at",
                current_timestamp()) \
    .dropDuplicates(["team_id", "_league"])

standings_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save(SILVER_STANDINGS)

log("silver_standings", "SUCCESS",
    "Standings cleaned",
    standings_silver.count())
print(f"  Silver standings: {standings_silver.count()}")
standings_silver.show(10)

print("\n" + "="*55)
print("03_SILVER LAYER COMPLETE ✅")
print("="*55)

In [0]:
# ============================================
# 04_GOLD LAYER
# Business metrics + KPIs
#
# Gold 1: League Standings   → Z-ORDER
# Gold 2: Team Performance   → Liquid Clustering
# Gold 3: Match Summary      → Partitioned
# Gold 4: Daily MIS          → Partitioned
# ============================================


from pyspark.sql.functions import *
from pyspark.sql.window import Window
import datetime

batch_id   = datetime.datetime.now() \
    .strftime("%Y%m%d%H%M%S")
batch_date = datetime.datetime.now() \
    .strftime("%Y-%m-%d")

print("="*55)
print("04_GOLD LAYER")
print("="*55)

def log(task, status, msg, records=0):
    data = [(task, status, msg, records,
             batch_id, batch_date,
             datetime.datetime.now()
             .strftime("%Y-%m-%d %H:%M:%S"))]
    spark.createDataFrame(
        data,
        ["task","status","message",
         "records","batch_id",
         "batch_date","log_time"]
    ).write \
     .format("delta") \
     .mode("append") \
     .save(LOG_PATH)
    print(f"  [{status}] {msg}")

# Read silver
events    = spark.read \
    .format("delta").load(SILVER_EVENTS)
standings = spark.read \
    .format("delta").load(SILVER_STANDINGS)

print(f"Silver events:    {events.count()}")
print(f"Silver standings: {standings.count()}")

In [0]:
%sql
select * from delta.`/Volumes/workspace/default/gangadhar_test/sports_analytics/silver/standings/`

In [0]:
# ── Gold 1: League Standings (Z-ORDER) ────────
print("\n1. League Standings (Z-ORDER)...")

w_rank = Window \
    .partitionBy("_league") \
    .orderBy(desc("points"),
             desc("goal_difference"))

league_standings = standings \
    .withColumn("position",
                row_number().over(w_rank)) \
    .withColumn("position_change",
        when(col("position") <= 4,
             "Champions League")
        .when(col("position") <= 6,
             "Europa League")
        .when(col("position") >= 18,
             "Relegation Zone")
        .otherwise("Mid Table")
    ) \
    .withColumn("_processed_at",
                current_timestamp())

league_standings.write \
    .format("delta") \
    .mode("overwrite") \
    .save(GOLD_LEAGUE_STANDINGS)

# Apply Z-ORDER
spark.sql(f"""
    OPTIMIZE delta.`{GOLD_LEAGUE_STANDINGS}`
    ZORDER BY (team_name, _league)
""")

print(f"  Records: {league_standings.count()} ✅")
print("  Z-ORDER applied ✅")
league_standings \
    .select(
        "position", "team_name",
        "played", "wins", "draws",
        "losses", "points",
        "position_change", "_league"
    ) \
    .orderBy("_league", "position") \
    .show(20, truncate=False)

log("gold_standings", "SUCCESS",
    "League standings with Z-ORDER",
    league_standings.count())

In [0]:
# ── Gold 2: Team Performance (Liquid) ─────────
print("\n2. Team Performance (Liquid Clustering)...")

# Home performance
home_stats = events \
    .filter(col("home_score").isNotNull()) \
    .groupBy("home_team", "_league") \
    .agg(
        count("event_id").alias("home_games"),
        sum(when(col("result") == "HOME_WIN", 1)
            .otherwise(0)).alias("home_wins"),
        sum("home_score").alias("home_goals_scored"),
        sum("away_score").alias("home_goals_conceded"),
        avg("home_score").alias("avg_home_goals")
    )

# Away performance
away_stats = events \
    .filter(col("away_score").isNotNull()) \
    .groupBy("away_team", "_league") \
    .agg(
        count("event_id").alias("away_games"),
        sum(when(col("result") == "AWAY_WIN", 1)
            .otherwise(0)).alias("away_wins"),
        sum("away_score").alias("away_goals_scored"),
        sum("home_score").alias("away_goals_conceded"),
        avg("away_score").alias("avg_away_goals")
    )

# Combine home + away
team_performance = home_stats \
    .join(
        away_stats,
        home_stats.home_team ==
        away_stats.away_team,
        how = "outer"
    ) \
    .withColumn("team_name",
        coalesce(col("home_team"),
                 col("away_team"))) \
    .withColumn("league",
        coalesce(home_stats["_league"],
                 away_stats["_league"])) \
    .withColumn("total_games",
        coalesce(col("home_games"), lit(0)) +
        coalesce(col("away_games"), lit(0))
    ) \
    .withColumn("total_wins",
        coalesce(col("home_wins"), lit(0)) +
        coalesce(col("away_wins"), lit(0))
    ) \
    .withColumn("total_goals_scored",
        coalesce(col("home_goals_scored"), lit(0)) +
        coalesce(col("away_goals_scored"), lit(0))
    ) \
    .withColumn("win_percentage",
        when(col("total_games") > 0,
             round(col("total_wins") /
                   col("total_games") * 100, 2))
        .otherwise(0)
    ) \
    .withColumn("_processed_at",
                current_timestamp()) \
    .select(
        "team_name", "league",
        "total_games", "total_wins",
        "total_goals_scored",
        "win_percentage",
        "avg_home_goals", "avg_away_goals",
        "_processed_at"
    )

# Create with Liquid Clustering
team_performance.limit(0).write \
    .format("delta") \
    .mode("overwrite") \
    .option("clusterBy", "team_name,league") \
    .save(GOLD_TEAM_PERFORMANCE)

team_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .save(GOLD_TEAM_PERFORMANCE)

spark.sql(f"""
    OPTIMIZE delta.`{GOLD_TEAM_PERFORMANCE}`
""")

print(f"  Records: {team_performance.count()} ✅")
print("  Liquid Clustering applied ✅")
team_performance \
    .orderBy(desc("win_percentage")) \
    .show(10, truncate=False)

log("gold_team_performance", "SUCCESS",
    "Team performance with Liquid Clustering",
    team_performance.count())

In [0]:
# ── Gold 3: Match Summary ─────────────────────
print("\n3. Match Summary...")

match_summary = events \
    .filter(col("home_score").isNotNull()) \
    .groupBy(
        "event_date",
        "_league",
        "result"
    ) \
    .agg(
        count("event_id")
        .alias("match_count"),
        sum("total_goals")
        .alias("total_goals"),
        avg("total_goals")
        .alias("avg_goals_per_match"),
        sum(when(col("is_high_scoring"),
            1).otherwise(0))
        .alias("high_scoring_matches")
    ) \
    .withColumn("_processed_at",
                current_timestamp())

match_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("_league", "event_date") \
    .save(GOLD_MATCH_SUMMARY)

print(f"  Records: {match_summary.count()} ✅")
match_summary.show(10)

# ── Gold 4: Daily MIS ─────────────────────────
print("\n4. Daily MIS...")

daily_mis = events \
    .filter(col("home_score").isNotNull()) \
    .groupBy("event_date", "_league") \
    .agg(
        count("event_id")
        .alias("total_matches"),
        sum("total_goals")
        .alias("total_goals"),
        avg("total_goals")
        .alias("avg_goals"),
        sum(when(col("result") == "HOME_WIN", 1)
            .otherwise(0))
        .alias("home_wins"),
        sum(when(col("result") == "AWAY_WIN", 1)
            .otherwise(0))
        .alias("away_wins"),
        sum(when(col("result") == "DRAW", 1)
            .otherwise(0))
        .alias("draws"),
        sum(when(col("is_high_scoring"), 1)
            .otherwise(0))
        .alias("high_scoring")
    ) \
    .withColumn("home_win_rate",
        round(col("home_wins") /
              col("total_matches") * 100, 2)
    ) \
    .withColumn("_processed_at",
                current_timestamp())

daily_mis.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("event_date") \
    .save(GOLD_DAILY_MIS)

print(f"  MIS records: {daily_mis.count()} ✅")
daily_mis.show(10)

log("gold_layer", "COMPLETE",
    "All gold tables ready")

print("\n" + "="*55)
print("04_GOLD LAYER COMPLETE ✅")
print(f"""
  Gold tables:
    League Standings  → Z-ORDER ✅
    Team Performance  → Liquid Clustering ✅
    Match Summary     → Partitioned ✅
    Daily MIS         → Partitioned ✅
""")
print("="*55)

In [0]:
# ============================================
# 05_STREAMING
# Auto process new JSON files
# as they land in landing zone
# Simulates real time processing
# ============================================


from pyspark.sql.functions import *
from pyspark.sql.types import *

print("="*55)
print("05_STREAMING — AUTO LOADER")
print("="*55)

# ── Schema for events ─────────────────────────
event_schema = StructType([
    StructField("idEvent",      StringType(), True),
    StructField("strEvent",     StringType(), True),
    StructField("strLeague",    StringType(), True),
    StructField("strHomeTeam",  StringType(), True),
    StructField("strAwayTeam",  StringType(), True),
    StructField("intHomeScore", StringType(), True),
    StructField("intAwayScore", StringType(), True),
    StructField("dateEvent",    StringType(), True),
    StructField("strTime",      StringType(), True),
    StructField("strVenue",     StringType(), True),
    StructField("strStatus",    StringType(), True),
])

# ── Auto Loader — watch landing zone ──────────
print("\nStarting Auto Loader stream...")
print(f"Watching: {LANDING_EVENTS}")

stream_df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.schemaLocation",
            f"{BASE}/schema_hints/events") \
    .option("multiLine", "true") \
    .load(LANDING_EVENTS)

# Transform stream
stream_transformed = stream_df \
    .withColumn("_stream_time",
                current_timestamp()) \
    .withColumn("_source_file",
                col("_metadata.file_path"))

# Write to Bronze Delta
stream_query = stream_transformed \
    .writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation",
            f"{BASE}/checkpoints/events") \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .start(BRONZE_EVENTS)

print("Stream started ✅")
print("New JSON files will be auto processed")
print("every 30 seconds")

# Show stream status
print(f"\nStream status: {stream_query.status}")
print(f"Stream ID: {stream_query.id}")

# Run for 60 seconds then stop
import time
time.sleep(60)
stream_query.stop()
print("\nStream stopped ✅")

In [0]:
# ============================================
# 06_REPORTS
# Final business reports
# ============================================

from pyspark.sql.functions import *
from pyspark.sql.window import Window

print("="*55)
print("SPORTS ANALYTICS — REPORTS")
print("="*55)

# ── Report 1: League Table ────────────────────
print("\n" + "="*55)
print("REPORT 1: EPL LEAGUE TABLE")
print("="*55)

spark.read \
    .format("delta") \
    .load(GOLD_LEAGUE_STANDINGS) \
    .filter(col("_league") == "EPL") \
    .select(
        "position", "team_name",
        "played", "wins", "draws",
        "losses", "goals_for",
        "goals_against", "goal_difference",
        "points", "position_change"
    ) \
    .orderBy("position") \
    .show(20, truncate=False)

# ── Report 2: Top performing teams ────────────
print("\n" + "="*55)
print("REPORT 2: TOP 5 PERFORMING TEAMS")
print("="*55)

spark.read \
    .format("delta") \
    .load(GOLD_TEAM_PERFORMANCE) \
    .select(
        "team_name", "league",
        "total_games", "total_wins",
        "total_goals_scored",
        "win_percentage"
    ) \
    .orderBy(desc("win_percentage")) \
    .limit(5) \
    .show(truncate=False)

# ── Report 3: High scoring matches ────────────
print("\n" + "="*55)
print("REPORT 3: HIGHEST SCORING MATCHES")
print("="*55)

spark.read \
    .format("delta") \
    .load(SILVER_EVENTS) \
    .filter(col("total_goals") >= 4) \
    .select(
        "event_date", "home_team",
        "home_score", "away_score",
        "away_team", "total_goals",
        "_league"
    ) \
    .orderBy(desc("total_goals")) \
    .limit(10) \
    .show(truncate=False)

# ── Report 4: Result distribution ─────────────
print("\n" + "="*55)
print("REPORT 4: RESULT DISTRIBUTION BY LEAGUE")
print("="*55)

spark.read \
    .format("delta") \
    .load(SILVER_EVENTS) \
    .filter(col("result") != "UNKNOWN") \
    .groupBy("_league", "result") \
    .agg(
        count("event_id").alias("count"),
        round(avg("total_goals"), 2)
        .alias("avg_goals")
    ) \
    .orderBy("_league", "result") \
    .show(truncate=False)

# ── Report 5: Daily MIS ───────────────────────
print("\n" + "="*55)
print("REPORT 5: DAILY MATCH SUMMARY")
print("="*55)

spark.read \
    .format("delta") \
    .load(GOLD_DAILY_MIS) \
    .orderBy(desc("event_date")) \
    .limit(10) \
    .show(truncate=False)

# ── Report 6: Pipeline logs ───────────────────
print("\n" + "="*55)
print("REPORT 6: PIPELINE AUDIT LOGS")
print("="*55)

spark.read \
    .format("delta") \
    .load(LOG_PATH) \
    .orderBy("log_time") \
    .show(truncate=False)

print("\n" + "="*55)
print("ALL REPORTS COMPLETE ✅")
print("="*55)

In [0]:
# ============================================
# 07_RUN_PIPELINE
# Orchestrate entire pipeline
# ============================================

import datetime
import time

print("="*55)
print("SPORTS ANALYTICS — FULL PIPELINE RUN")
print(f"Started: {datetime.datetime.now()}")
print("="*55)

pipeline_start = time.time()

steps = [
    "01_fetch_api_data",
    "02_bronze",
    "03_silver",
    "04_gold",
    "06_reports",
]

for notebook in steps:
    print(f"\nRunning: {notebook}...")
    t = time.time()
    try:
        dbutils.notebook.run(
            notebook,
            timeout_seconds=300,
            arguments={}
        )
        elapsed = time.time() - t
        print(f"  {notebook} → ✅ ({elapsed:.1f}s)")
    except Exception as e:
        print(f"  {notebook} → ❌ FAILED")
        print(f"  Error: {str(e)}")
        raise e

total = time.time() - pipeline_start

print("\n" + "="*55)
print("PIPELINE COMPLETE ✅")
print(f"  Total time: {total:.1f}s")
print(f"  Completed:  {datetime.datetime.now()}")
print("="*55)